In [0]:
import pandas as pd

query = """
SELECT
    c.city,
    c.region,
    d.date,
    d.year,
    d.month,
    d.season,
    f.pm25, f.pm10, f.no2, f.so2, f.co, f.o3, f.aqi
FROM workspace.aqi.fact_aqi_daily f
JOIN workspace.aqi.dim_city c ON f.city_id = c.city_id
JOIN workspace.aqi.dim_date d ON f.date_id = d.date_id
WHERE NOT (c.city = 'Ahmedabad' AND f.aqi_more_500 = true)
"""

df = spark.sql(query).toPandas()
print("Rows:", len(df))
print("Columns:", list(df.columns))
df.head()

Rows: 24438
Columns: ['city', 'region', 'date', 'year', 'month', 'season', 'pm25', 'pm10', 'no2', 'so2', 'co', 'o3', 'aqi']


,city,region,date,year,month,season,pm25,pm10,no2,so2,co,o3,aqi
0,Ahmedabad,West,2015-01-29,2015,1,Winter,83.13,NaN,28.71,49.52,NaN,59.76,209.0
1,Ahmedabad,West,2015-01-30,2015,1,Winter,79.84,NaN,28.68,48.49,NaN,97.07,328.0
2,Ahmedabad,West,2015-02-04,2015,2,Winter,80.65,NaN,22.83,25.73,NaN,47.30,294.0
3,Ahmedabad,West,2015-02-05,2015,2,Winter,58.36,NaN,21.39,32.66,NaN,53.54,149.0
4,Ahmedabad,West,2015-02-06,2015,2,Winter,79.29,NaN,26.94,67.41,NaN,59.30,190.0


In [0]:
pollutants = ['pm25', 'pm10', 'no2', 'so2', 'co', 'o3', 'aqi']

df[pollutants].describe().round(1)

,pm25,pm10,no2,so2,co,o3,aqi
count,23760.0,17682.0,24047.0,23838.0,23159.0,23661.0,24438.0
mean,67.1,118.3,27.9,13.4,1.2,34.8,155.6
std,63.3,89.6,22.5,15.0,1.7,21.7,106.7
min,0.0,0.0,0.0,0.0,0.0,0.0,13.0
25%,28.7,56.6,11.8,5.7,0.6,19.1,81.0
50%,48.2,95.9,21.7,9.1,0.9,31.0,116.0
75%,79.8,149.8,37.3,14.7,1.4,45.8,199.0
max,914.9,917.1,362.2,186.1,48.1,257.7,956.0


In [0]:
corr = df[pollutants].corr(method = "spearman").round(2)
corr

,pm25,pm10,no2,so2,co,o3,aqi
pm25,1.00,0.88,0.46,0.27,0.39,0.27,0.85
pm10,0.88,1.00,0.52,0.40,0.34,0.28,0.87
no2,0.46,0.52,1.00,0.23,0.24,0.31,0.44
so2,0.27,0.40,0.23,1.00,0.16,0.19,0.35
co,0.39,0.34,0.24,0.16,1.00,0.07,0.49
o3,0.27,0.28,0.31,0.19,0.07,1.00,0.27
aqi,0.85,0.87,0.44,0.35,0.49,0.27,1.00


In [0]:
def count_outliers(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    upper = q3 + 3 * iqr
    return pd.Series({
        "q1": round(q1, 1),
        "q3": round(q3, 1),
        "upper_limit": round(upper, 1),
        "outliers": int((series > upper). sum()),
        "pct_outliers": round((series > upper).mean() * 100, 2),
        "zeros": int((series == 0).sum()),
    })
outliers = df[pollutants].apply(lambda col: count_outliers(col.dropna())).T
outliers

,q1,q3,upper_limit,outliers,pct_outliers,zeros
pm25,28.7,79.8,232.8,718.0,3.02,0.0
pm10,56.6,149.8,429.4,200.0,1.13,0.0
no2,11.8,37.3,113.9,188.0,0.78,0.0
so2,5.7,14.7,41.7,1148.0,4.82,0.0
co,0.6,1.4,3.7,574.0,2.48,773.0
o3,19.1,45.8,125.8,83.0,0.35,0.0
aqi,81.0,199.0,553.0,59.0,0.24,0.0


In [0]:
def outlier_cities(column, upper):
    flagged = df[df[column] > upper]
    return (flagged.groupby("city")
                    .size()
                    .sort_values(ascending = False)
                    .head(5))
print("SO2 outliers by city:")
print(outlier_cities("so2", 41.7))
print()
print("PM2.5 outliers by city:")
print(outlier_cities("pm25", 232.8))
print()
print("PM2.5 outliers by month:")
print(df[df["pm25"] > 232.8].groupby("month").size())


SO2 outliers by city:
city
Ahmedabad     407
Patna         349
Jorapokhar    179
Talcher       119
Delhi          31
dtype: int64

PM2.5 outliers by city:
city
Patna       222
Delhi       188
Lucknow     147
Gurugram    112
Kolkata      17
dtype: int64

PM2.5 outliers by month:
month
1     186
2      26
3      14
4       6
5      13
6      22
7      11
8       2
10     32
11    202
12    206
dtype: int64


In [0]:
from scipy import stats

# Same dates (25 Mar - 31 May) in 2019 and 2020
period = df[
    (df["date"].astype(str).str[5:] >= "03-25") &
    (df["date"].astype(str).str[5:] <= "05-31") &
    (df["year"].isin([2019, 2020]))
]

#Average PM2.5 per city per year, side by side
city_year = period.pivot_table(index = "city", columns = "year", values = "pm25", aggfunc = "mean").dropna()
city_year.columns = ["pm25_2019", "pm25_2020"]
city_year["pct_change"] = ((city_year["pm25_2020"] - city_year["pm25_2019"]) / city_year['pm25_2019'] * 100).round(1)

print("Cities with data in both years:", len(city_year))
print("Median change:", city_year["pct_change"].median(), "%")

#Wilcoxon signed-rank test: did PM2.5 fall consistently across cities?
result = stats.wilcoxon(city_year["pm25_2019"], city_year["pm25_2020"], alternative = 'greater')
print("Wilcoxon statistic:", result.statistic)
print("p-value:", result.pvalue)

city_year.sort_values("pct_change")

Cities with data in both years: 19
Median change: -38.2 %
Wilcoxon statistic: 190.0
p-value: 1.9073486328125e-06


,pm25_2019,pm25_2020,pct_change
city,,,
Ahmedabad,67.382857,28.880588,-57.1
Chennai,39.717794,18.434559,-53.6
Gurugram,91.415588,45.543088,-50.2
Amritsar,52.575000,26.280323,-50.0
Kolkata,45.194559,22.728971,-49.7
Bengaluru,45.714853,23.464853,-48.7
Visakhapatnam,34.776000,18.144615,-47.8
Delhi,86.193529,48.481912,-43.8
Lucknow,93.905147,55.378088,-41.0
